In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [2]:
wine = load_wine()
X = wine.data
y_true = wine.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Размерность данных после масштабирования: {X_scaled.shape}")

Размерность данных после масштабирования: (178, 13)


In [3]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
db_preds = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_preds)) - (1 if -1 in db_preds else 0)
n_noise_db = list(db_preds).count(-1)

print("--- DBSCAN (eps=1.5, min_samples=5) ---")
print(f"Количество кластеров (без учета шума): {n_clusters_db}")
print(f"Количество выбросов (шума): {n_noise_db}")

if n_clusters_db > 1:
    mask = db_preds != -1
    sil_db = silhouette_score(X_scaled[mask], db_preds[mask])
    ari_db = adjusted_rand_score(y_true, db_preds)
    print(f"Silhouette Score: {sil_db:.4f}")
    print(f"Adjusted Rand Score (ARI): {ari_db:.4f}")

--- DBSCAN (eps=1.5, min_samples=5) ---
Количество кластеров (без учета шума): 3
Количество выбросов (шума): 6
Silhouette Score: 0.2874
Adjusted Rand Score (ARI): 0.3685


In [4]:
kmeans = KMeans(n_clusters=3, random_state=100, n_init=10)
km_preds = kmeans.fit_predict(X_scaled)

sil_km = silhouette_score(X_scaled, km_preds)
ari_km = adjusted_rand_score(y_true, km_preds)

print("--- KMeans (n_clusters=3) ---")
print(f"Silhouette Score: {sil_km:.4f}")
print(f"Adjusted Rand Score (ARI): {ari_km:.4f}")

--- KMeans (n_clusters=3) ---
Silhouette Score: 0.2849
Adjusted Rand Score (ARI): 0.8991


## Выводы и сравнение:

1. **Качество кластеризации (ARI)**:
   - **KMeans** показал значительно более высокий результат `Adjusted Rand Score` (~0.90), что говорит о почти идеальном совпадении найденных кластеров с истинными сортами вина в датасете.
   - **DBSCAN** справился хуже (ARI ~0.37), так как алгоритму на основе плотности сложно подобрать единые параметры `eps` и `min_samples` для многомерных данных винограда, из-за чего часть объектов ушла в шумовые выбросы.

2. **Силуэт (Silhouette Score)**:
   - Значения силуэта у обоих алгоритмов сопоставимы (~0.28), однако KMeans разделяет все объекты сплошным образом по центроидам, в то время как DBSCAN выделяет плотные зоны, оставляя часть точек за пределами кластеров.